# 04 · Persistence and memory: surviving restarts, remembering accounts

**You'll learn:** how the SQLite checkpointer lets a paused review survive a full restart, how to list pending reviews and step through a run's history, how long-term account memory changes the next decision, and where to watch all of this in Studio, the app and the CLI.

**Time:** about 10 minutes · **Cost:** free tier or about 1¢ · **Needs:** an API key in `.env`.

```mermaid
flowchart LR
  subgraph RUN["One run (thread_id)"]
    direction LR
    A["triage → specialists → aggregator"] --> P["paused before human_review"]
    P -->|"update_state + invoke(None)"| R["resolution"]
  end
  P -.->|"every step saved"| CP[("checkpoints.sqlite<br/>SqliteSaver")]
  R -.->|"decision saved"| ST[("account_memory.sqlite<br/>SqliteStore")]
  ST -.->|"read by triage next time"| A
```

| Part | What it covers | Code |
|---|---|---|
| 1 | Setup | |
| 2 | Pause a review, then "restart" | `graph.py`, `langgraph.checkpoint.sqlite` |
| 3 | Pending reviews and run history | `ui/logic.py` |
| 4 | Long-term memory changes the next decision | `nodes/resolution.py`, `nodes/triage.py` |
| 5 | Studio, the app and the CLI | |
| 6 | Recap and things to try | |

---
## Part 1 · Setup

### Setup
The next cell connects the notebook to the project. It gives this notebook **its own** database files in `.notebook_state/`, so nothing here touches the Streamlit app's data. It prints which LLM this run uses and whether its key is set (never the key itself). Set `SHOW_LIVE_LOGS = True` to see each node's live log line.

In [ ]:
# ── Settings ────────────────────────────────────────────────────────────
SHOW_LIVE_LOGS = False  # True = print every node's live log line, like the terminal does

# Setup: run from the repo root, and give this notebook its own database files in
# .notebook_state/ so it never touches the app's checkpoints.sqlite / account_memory.sqlite.
import logging
import os
import sys
import warnings
from pathlib import Path

if "google.colab" in sys.modules:
    # Google Colab: the notebook needs the whole repo (it imports core/, agents/, tools/),
    # so clone it and install the pinned packages on first run. Re-running skips this.
    import subprocess

    REPO = Path("/content/fintech-agent")
    if not REPO.exists():
        print("Colab: cloning the repo and installing packages (about a minute)...")
        subprocess.run(["git", "clone", "-q", "-b", "develop-gemini",
                        "https://github.com/iam-suraj-kumar/fintech-agent.git", str(REPO)], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)
        print("Done. If a later cell fails to import, use Runtime > Restart session and run this cell again.")
    os.chdir(REPO)
    try:
        from google.colab import userdata  # add GEMINI_API_KEY under the key icon (Secrets) in the sidebar

        os.environ.setdefault("GEMINI_API_KEY", userdata.get("GEMINI_API_KEY"))
    except Exception:
        print("No GEMINI_API_KEY secret found: notebooks 00 and 01 still work; 02-04 need it.")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
STATE_DIR = ROOT / ".notebook_state"
STATE_DIR.mkdir(exist_ok=True)
os.environ["CHECKPOINT_DB_PATH"] = str(STATE_DIR / "04_checkpoints.sqlite")
os.environ["MEMORY_DB_PATH"] = str(STATE_DIR / "04_memory.sqlite")
os.environ.setdefault("FASTMCP_LOG_LEVEL", "WARNING")  # quiet the MCP server subprocess
warnings.filterwarnings("ignore")

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")  # never overrides the two database paths set above

import pandas as pd

pd.set_option("display.max_colwidth", None)  # show agent findings in full, not cut off with "..."


def trace_table(audit_log: list[dict]) -> pd.DataFrame:
    """One readable line per audit entry (entries have different fields, so a raw table is mostly blanks)."""
    def detail(e: dict) -> str:
        if e["kind"] == "tool_call":
            return f"calls {e['tool']}({e['args']})"
        if e["kind"] == "tool_result":
            return f"{e['tool']} returned {e['result']}"
        return str(e.get("text", ""))
    return pd.DataFrame([{"node": e.get("node", ""), "kind": e["kind"], "detail": detail(e)} for e in audit_log])

import core.audit  # noqa: F401  (creates the "fraud_guard" live logger)
from core.llm import llm_model, llm_provider

logging.getLogger("fraud_guard").setLevel(logging.INFO if SHOW_LIVE_LOGS else logging.WARNING)
pd.set_option("display.max_colwidth", 160)
key = {"gemini": "GEMINI_API_KEY", "openai": "OPENAI_API_KEY"}.get(llm_provider(), "?")
print(f"LLM: {llm_provider()} / {llm_model()}   ({key} set: {'yes' if os.environ.get(key) else 'NO'})")

import sqlite3
import uuid

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.store.sqlite import SqliteStore

from core.state import new_state
from data.synthetic_transactions import SCENARIOS
from graph import DEFAULT_NODES, build_graph, graph

CHECKPOINTS = os.environ["CHECKPOINT_DB_PATH"]
MEMORY = os.environ["MEMORY_DB_PATH"]
print("checkpoints:", CHECKPOINTS, "\nmemory:     ", MEMORY)

---
## Part 2 · Pause a review, then "restart"

Start the $9,500 wire to "Silk Road Holdings Ltd". It pauses before `human_review`, and every step is already in `checkpoints.sqlite`.

In [ ]:
config = {"configurable": {"thread_id": f"restart-demo-{uuid.uuid4().hex[:6]}"}}
graph.invoke(new_state(SCENARIOS["borderline_restart_demo"]), config)
print("waiting before:", graph.get_state(config).next, "· thread:", config["configurable"]["thread_id"])

Now simulate killing the process: throw away the graph object and build a **brand-new** one from nothing but the same two files on disk. Nothing in memory is reused.

In [ ]:
del graph  # the old graph object is gone

fresh_graph = build_graph(
    DEFAULT_NODES,
    checkpointer=SqliteSaver(sqlite3.connect(CHECKPOINTS, check_same_thread=False)),
    store=SqliteStore(sqlite3.connect(MEMORY, check_same_thread=False, isolation_level=None)),
)
restored = fresh_graph.get_state(config)
print("after restart, still waiting before:", restored.next)
print(f"composite score {restored.values['composite_score']:.2f} · analyst: {restored.values['analyst_findings'][:160]}…")

**What to notice:** the pending case, with all three findings, came back from disk. A server deploy or crash in the middle of a review loses nothing.

---
## Part 3 · Pending reviews and run history

The app's review queue is just "every thread whose next step is `human_review`" (`ui/logic.py: get_pending_reviews()`):

In [ ]:
from ui.logic import get_pending_reviews

pd.DataFrame([
    {"thread": p["thread_id"], "amount": p["transaction"]["amount"], "composite": round(p["composite_score"], 2)}
    for p in get_pending_reviews(fresh_graph)
])

The checkpointer keeps **every** step, not just the last, so you can see how the state evolved (and, in Studio, rewind to any step):

In [ ]:
history = list(fresh_graph.get_state_history(config))
pd.DataFrame([
    {"step": s.metadata.get("step"), "next node": s.next, "audit entries so far": len(s.values.get("audit_log", []))}
    for s in reversed(history)
])

The reviewer escalates it. `resolution` records the decision in long-term memory:

In [ ]:
fresh_graph.update_state(config, {"human_decision": "escalate"})
print("final decision:", fresh_graph.invoke(None, config)["final_decision"])

---
## Part 4 · Long-term memory changes the next decision

The checkpointer remembered **one run**. The store remembers **the account** across runs. `borderline_wire` uses the same account, `acct_2002`, which now has one escalation on record:

In [ ]:
account = SCENARIOS["borderline_wire"]["account_id"]
store = SqliteStore(sqlite3.connect(MEMORY, check_same_thread=False, isolation_level=None))
memory = [item.value for item in store.search(("account_memory", account))]
print(account, "memory:", memory)
if not memory:
    print("(empty: run the escalate cell above first, or the wire is investigated as if new)")

wire = fresh_graph.invoke(new_state(SCENARIOS["borderline_wire"]), {"configurable": {"thread_id": f"wire-{uuid.uuid4().hex[:6]}"}})
print(f"\nborderline_wire triage score now {wire['triage_score']:.2f} (0.64 on a clean account) -> {wire['triage_decision']} -> {wire['final_decision']}")

**What to notice:** one escalation added +0.15 and pushed the wire past the 0.65 auto-decline threshold, so this time it was declined instantly with no LLM calls. The app wipes memory on every start by default (`RESET_MEMORY_ON_START=true`) so rehearsals don't snowball like this; set it to `false` to keep memory between runs.

---
## Part 5 · Studio, the app and the CLI

The same graph, three other ways in:

| Tool | Command | What you see |
|---|---|---|
| **LangGraph Studio** | `uv run langgraph dev` | the graph drawn live; run it, watch each node, inspect and rewind state |
| **Streamlit app** | `uv run streamlit run ui/app.py` | submit transactions, the review queue with Approve / Decline / Escalate, history |
| **CLI** | `uv run python cli.py submit borderline_wire` | the same flow in the terminal; `pending`, `review <thread_id> approve`, `history` |
| **LangSmith** (optional) | `LANGSMITH_TRACING=true` in `.env` | every LLM call, tool call and timing, per run |

---
## Part 6 · Recap and things to try

**What happened:** a paused review survived throwing away the whole graph, because every step lives in SQLite; the review queue and step history came straight from the checkpointer; and a decision stored in long-term memory changed how the same account was triaged next time.

**Try this**
- Decide the restored case with `"approve"` instead: does `borderline_wire` still get declined in Part 4?
- Run Part 2, then actually restart the kernel (**Kernel → Restart**), re-run Setup, and look the thread up again with `get_pending_reviews`.
- Open Studio (`uv run langgraph dev`) and run `borderline_wire` there.